In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import sklearn as sk

In [46]:
df_customers_real = pd.read_csv("data/olist_customers_dataset.csv")
df_geolocation_real = pd.read_csv("data/olist_geolocation_dataset.csv")
df_order_items_real = pd.read_csv("data/olist_order_items_dataset.csv")
df_order_payments_real = pd.read_csv("data/olist_order_payments_dataset.csv")
df_order_reviews_real = pd.read_csv("data/olist_order_reviews_dataset.csv")
df_orders_real = pd.read_csv("data/olist_orders_dataset.csv")
df_products_real = pd.read_csv("data/olist_products_dataset.csv")
df_sellers_real = pd.read_csv("data/olist_sellers_dataset.csv")
df_translation_real = pd.read_csv("data/product_category_name_translation.csv")

In [47]:
df_customers = df_customers_real.copy()
df_geolocation = df_geolocation_real.copy()
df_order_items = df_order_items_real.copy()
df_order_payments = df_order_payments_real.copy()
df_order_reviews = df_order_reviews_real.copy()
df_orders = df_orders_real.copy()
df_products = df_products_real.copy()
df_sellers = df_sellers_real.copy()
df_translation = df_translation_real.copy()

# Reviews

In [48]:
# reviews 테이블 null 값 확인
df_order_reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [49]:
df_order_reviews.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [50]:
# 날짜 타입 변환
df_order_reviews['review_creation_date']    = pd.to_datetime(df_order_reviews['review_creation_date'])
df_order_reviews['review_answer_timestamp'] = pd.to_datetime(df_order_reviews['review_answer_timestamp'])

print(df_order_reviews[['review_creation_date', 'review_answer_timestamp']].dtypes)

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [51]:
# review_comment_title, review_comment_message 컬럼 제거
# 컬럼 제거
df_order_reviews_clean = df_order_reviews.drop(
    columns=['review_comment_title', 'review_comment_message']
)
print(df_order_reviews_clean.columns.tolist())

['review_id', 'order_id', 'review_score', 'review_creation_date', 'review_answer_timestamp']


In [52]:
df_order_reviews_clean.head()

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01,2018-03-02 10:26:53


### reveiw_score 이상치 확인

In [53]:
df_order_reviews_clean.value_counts('review_score')

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

### order_id 중복 확인

Olist는 주문 단위 리뷰이므로 order_id가 중복인 경우는 불가능한 것이 맞음

In [54]:
# 중복 order_id 확인
dup_count = df_order_reviews_clean.duplicated('order_id').sum()
total = len(df_order_reviews_clean)
print(f"전체 리뷰: {total:,}건")
print(f"중복 order_id: {dup_count:,}건")
print(f"고유 order_id: {df_order_reviews_clean['order_id'].nunique():,}건")

전체 리뷰: 99,224건
중복 order_id: 551건
고유 order_id: 98,673건


중복 order_id에 대한 리뷰 예시

2018-03-21 00:00:00
"이 주문에서 구매한 제품 중 하나(배송 02)는 아직 받지 못했고, 다른 하나(배송 01)는 배송에 어려움이 있다는 이메일을 받은 뒤로 더 이상 소식이 없습니다.

2018-03-29 00:00:00
"이미 여러 차례 항의했지만, 제가 구매한 제품이 어디에 있는지 오늘까지도 알 수 없습니다. 이 구매를 취소하고 환불받기를 원합니다."

고객의 최종 의사가 반영된 최근 것만 남겨두는 것이 나중에 merge 할 때 데이터가 늘어나는 것을 막을 수 있을 것 같음 

In [55]:
# 가장 최신 리뷰만 유지 (review_creation_date 기준)
df_order_reviews_clean = (
    df_order_reviews_clean
    .sort_values('review_creation_date')
    .drop_duplicates('order_id', keep='last')  # 마지막(최신) 리뷰만 유지
    .reset_index(drop=True)
)

print(f"중복 제거 후: {len(df_order_reviews_clean):,}행")
print(f"고유 order_id 수: {df_order_reviews_clean['order_id'].nunique():,}개")

중복 제거 후: 98,673행
고유 order_id 수: 98,673개


# Orders

In [56]:
# custo = pd.read_csv('../data/olist_customers_dataset.csv')
# custo_copy = custo.copy()
# custo_copy.info()
# custo_copy.isnull().sum()  
# #99441 entries, Data columns (total 5 columns)

# #customer_unique_id 칼럼 삭제 _ 타 table과 조인할 때도 customer_id를 기준으로 진행될 것
# #cusomer 개인 정보 보호를 위해서 더 찾아볼 수 있는 정보 없으므로 따로 사용할 가능성 낮음
# custo_copy = custo_copy.drop(columns=['customer_unique_id'])
# custo_copy.info()
# #99441 entries, Data columns (total 4 columns)

In [57]:
# pay = pd.read_csv('data/olist_order_payments_dataset.csv')
# pay.info()
# pay.isnull().sum()
# df_order_payments = pay.copy()
# df_order_payments
# review = pd.read_csv('data/olist_order_reviews_dataset.csv')
# review.info()
# review.isnull().sum()
#df_order_reviews = review.copy()

In [58]:
# order = pd.read_csv("data/olist_orders_dataset.csv")
# order_copy = order.copy()
# df_orders = order_copy

# order_copy.info()
# order_copy.isnull().sum()


#형변환

##결측X / (시각데이터 날림 코드 밑에 주석처리)
#date type 변환 (str -> date) / 연산 필요시 date 재변환 가능성 있음

#제거 전 확인
#time_part = order_copy['order_estimated_delivery_date'].str[-8:] 
#no_time = (time_part == '00:00:00').sum()
#print(no_time) #99441 총 행 갯수와 동일 모두 00:00:00 _제거 진행

#시분초 제거 및 타입 전환 코드 
# order_copy['order_estimated_delivery_date'] = pd.to_datetime(
#     order_copy['order_estimated_delivery_date'], 
#     format='%Y-%m-%d %H:%M:%S'
# ).dt.date
#order_copy['order_estimated_delivery_date'].head(20)

''' 시각 데이터 모든행 '00:00:00' '''
df_orders['order_estimated_delivery_date'] = pd.to_datetime(
    df_orders['order_estimated_delivery_date'], 
    format='%Y-%m-%d %H:%M:%S'
)

#형변환
##결측X 
#date type 변환 (str -> date) 
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])


#형변환 
##결측 O 
#date type 변환 (str -> date) 
cols_na = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

df_orders[cols_na] = df_orders[cols_na].apply(
    pd.to_datetime,
    errors="coerce"
)

#df_orders.info()


In [59]:
#데이터 정제- 불필요 행 제거

#order_status 칼럼 - 결측X, 분석 목표인 배달 요소와 관련 없는 status 분리하여 분석 진행이 용이할 것으로 판단
#'배송 품질' 일단 Olist와 직결된 물류 배송에 국한하여 정의/ 상품 준비단계에서 소요되는 날짜 논외
core_statuses = ['delivered', 'shipped', 'canceled']
df_orders = df_orders[df_orders['order_status'].isin(core_statuses)]
df_orders['order_status'].value_counts()

df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 155
order_delivered_carrier_date      552
order_delivered_customer_date    1734
order_estimated_delivery_date       0
dtype: int64

In [60]:
#결측 처리

# 1. 이상치 분리: 
# order_status 'canceled'인데 order_delivered_customer_date 값 있음 #6개 
# >> review message 보니까 제거하면 안되는 case, 새로운 데이터프레임으로 분리 저장
con_0 = (
    (df_orders["order_status"] == "canceled") &
    (df_orders["order_delivered_customer_date"].notna())
)
df_canceled_with_delivery = df_orders[con_0].copy()

# 2. df_orders에서는 해당 행들 삭제
df_orders = df_orders[~con_0]

# 3. 결과 확인
print(f"분리된 행: {df_canceled_with_delivery.shape[0]}개")
print(f"df_orders 남은 행: {df_orders.shape[0]}개")
print(f"\ndf_canceled_with_delivery:")
print(df_canceled_with_delivery)

df_orders.isnull().sum()

분리된 행: 6개
df_orders 남은 행: 98204개

df_canceled_with_delivery:
                               order_id                       customer_id  \
2921   1950d777989f6a877539f53795b4c3c3  1bccb206de9f0f25adc6871a1bcf77b2   
8791   dabf2b0e35b423f94618bf965fcb7514  5cdec0bb8cbdf53ffc8fdc212cd247c6   
58266  770d331c84e5b214bd9dc70a10b829d0  6c57e6119369185e575b36712766b0ef   
59332  8beb59392e21af5eb9547ae1a9938d06  bf609b5741f71697f65ce3852c5d2623   
92636  65d1e226dfaeb8cdc42f665422522d14  70fc57eeae292675927697fe03ad3ff5   
94399  2c45c33d2f9cb8ff8b1c86cc28c11c30  de4caa97afa80c8eeac2ff4c8da5b72e   

      order_status order_purchase_timestamp   order_approved_at  \
2921      canceled      2018-02-19 19:48:52 2018-02-19 20:56:05   
8791      canceled      2016-10-09 00:56:52 2016-10-09 13:36:58   
58266     canceled      2016-10-07 14:52:30 2016-10-07 15:07:10   
59332     canceled      2016-10-08 20:17:50 2016-10-09 14:34:30   
92636     canceled      2016-10-03 21:01:41 2016-10-04 10:18:57 

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 155
order_delivered_carrier_date      552
order_delivered_customer_date    1734
order_estimated_delivery_date       0
dtype: int64

In [61]:
# order_status 'canceled'인데 order_delivered_customer_date 값 있는 6개 데이터, 분리한 df
df_canceled_with_delivery

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [62]:
# 'delivered' 상태 행만 추출
#df_delivered = df_orders[df_orders["order_status"] == "delivered"].copy()
# 조회 확인
# print(df_delivered.head())
# print(df_delivered.isna().sum())
#order_approved_at                14
# order_delivered_carrier_date      2
# order_delivered_customer_date     8

# 결측 처리
# order_status 'delivered', 'order_approved_at'컬럼 결측 대체
# 조건 정의
con_1 = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_approved_at'].isna())
)

# 결측 order_purchase_timestamp로 대체
df_orders.loc[con_1, 'order_approved_at'] = df_orders.loc[con_1, 'order_purchase_timestamp']

# 결과 확인
print(con_1.sum()) #행갯수
print(df_orders.loc[con_1, ['order_approved_at', 'order_purchase_timestamp']].head())

14
        order_approved_at order_purchase_timestamp
5323  2017-02-18 14:40:00      2017-02-18 14:40:00
16567 2017-02-18 12:45:31      2017-02-18 12:45:31
19031 2017-02-18 13:29:47      2017-02-18 13:29:47
22663 2017-02-18 16:48:35      2017-02-18 16:48:35
23156 2017-02-17 13:05:55      2017-02-17 13:05:55


In [63]:
# order_status 'delivered', 'order_delivered_carrier_date'컬럼 결측 대체
# 조건 정의
con_2 = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_carrier_date'].isna())
)

# 해당 행들만 order_approved_at로 대체
df_orders.loc[con_2, 'order_delivered_carrier_date'] = df_orders.loc[con_2, 'order_approved_at']

# 결과 확인
print(con_2.sum()) #행갯수
print(df_orders.loc[con_2, ['order_delivered_carrier_date', 'order_approved_at']].head())

2
      order_delivered_carrier_date   order_approved_at
73222          2017-09-29 09:07:16 2017-09-29 09:07:16
92643          2017-05-25 23:30:16 2017-05-25 23:30:16


In [64]:
# order_status 'delivered', 'order_delivered_customer_date'컬럼 결측 reviews table 리뷰 작성 날짜로 대체
# df_orders와 df_order_reviews(order_id 기준) 임의 merge
df_orders = df_orders.merge(
    df_order_reviews_clean[['order_id', 'review_creation_date']],
    on='order_id',
    how='left'
)

# review_creation_date datetime 임의 변환
df_orders['review_creation_date'] = pd.to_datetime(df_orders['review_creation_date'], errors='coerce')

# 조건 정의
con_3 = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_customer_date'].isna())
)

# 결측 채우기
df_orders.loc[con_3, 'order_delivered_customer_date'] = df_orders.loc[con_3, 'review_creation_date']

# 결과 확인
print(con_3.sum()) #행갯수
print(df_orders.loc[con_3, ['order_delivered_customer_date', 'review_creation_date']].head())


8
      order_delivered_customer_date review_creation_date
2969                     2017-12-19           2017-12-19
20374                    2018-06-29           2018-06-29
43302                    2018-07-11           2018-07-11
78298                    2018-07-07           2018-07-07
81843                    2018-07-06           2018-07-06


In [65]:
# merge 후 불필요 컬럼 제거
df_orders = df_orders.drop(columns=['review_creation_date'])

# 제거 후 확인
print(df_orders.loc[con_3, ['order_delivered_customer_date']].head())

      order_delivered_customer_date
2969                     2017-12-19
20374                    2018-06-29
43302                    2018-07-11
78298                    2018-07-07
81843                    2018-07-06


In [66]:
df_orders.info()
df_orders.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 98204 entries, 0 to 98203
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       98204 non-null  str           
 1   customer_id                    98204 non-null  str           
 2   order_status                   98204 non-null  str           
 3   order_purchase_timestamp       98204 non-null  datetime64[us]
 4   order_approved_at              98063 non-null  datetime64[us]
 5   order_delivered_carrier_date   97654 non-null  datetime64[us]
 6   order_delivered_customer_date  96478 non-null  datetime64[us]
 7   order_estimated_delivery_date  98204 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.0 MB


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 141
order_delivered_carrier_date      550
order_delivered_customer_date    1726
order_estimated_delivery_date       0
dtype: int64

In [67]:
# 조건 정의: shipped 상태 & order_delivered_customer_date 결측 정상_ "2099-01-01 00:00:00" 형태로 대체
con_4= (
    (df_orders['order_status'] == 'shipped') &
    (df_orders['order_delivered_customer_date'].isna())
)

# 정상 결측 값 정의
normal_blank = pd.Timestamp("2099-01-01 00:00:00")

# 결측 대체
df_orders.loc[con_4, 'order_delivered_customer_date'] = normal_blank

# 결과 확인
print("채운 행 개수:", con_4.sum())
print(df_orders.loc[con_4, ['order_id', 'order_delivered_customer_date']].head())

채운 행 개수: 1107
                             order_id order_delivered_customer_date
43   ee64d42b8cf066f35eac1cf57de1aa85                    2099-01-01
151  6942b8da583c2f9957e990d028607019                    2099-01-01
159  36530871a5e80138db53bcfd8a104d90                    2099-01-01
228  4d630f57194f5aba1a3d12ce23e71cd9                    2099-01-01
295  3b4ad687e7e5190db827e1ae5a8989dd                    2099-01-01


In [68]:
# 조건 정의: canceled 상태 & order_delivered_customer_date 결측 정상_ "2099-01-01 00:00:00" 형태로 대체
# 조건: canceled 상태
con_5 = df_orders['order_status'] == 'canceled'

# 결측 처리할 컬럼 리스트
cols_fill= [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

# 한 번에 결측 처리
for col in cols_fill:
    df_orders.loc[con_5 & df_orders[col].isna(), col] = normal_blank

# 결과 확인
display(df_orders.loc[con_5, ['order_id', 'order_status'] + cols_fill].head())

,order_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
392,1b9ecfe83cdc259250e1a8aca174f0ad,canceled,2018-08-07 04:10:26,2099-01-01 00:00:00,2099-01-01
606,714fb133a6730ab81fa1d3c1b2007291,canceled,2018-01-26 21:58:39,2018-01-29 22:33:25,2099-01-01
1048,3a129877493c8189c59c60eb71d97c29,canceled,2018-01-25 13:50:20,2018-01-26 21:42:18,2099-01-01
1120,00b1cb0320190ca0daa2c88b35206009,canceled,2099-01-01 00:00:00,2099-01-01 00:00:00,2099-01-01
1784,ed3efbd3a87bea76c2812c66a0b32219,canceled,2099-01-01 00:00:00,2099-01-01 00:00:00,2099-01-01


In [69]:
df_orders.info()
df_orders.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 98204 entries, 0 to 98203
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       98204 non-null  str           
 1   customer_id                    98204 non-null  str           
 2   order_status                   98204 non-null  str           
 3   order_purchase_timestamp       98204 non-null  datetime64[us]
 4   order_approved_at              98204 non-null  datetime64[us]
 5   order_delivered_carrier_date   98204 non-null  datetime64[us]
 6   order_delivered_customer_date  98204 non-null  datetime64[us]
 7   order_estimated_delivery_date  98204 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.0 MB


order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

In [70]:
# ── 이상치 조건 정의 (논리적으로 역전된 경우) ──
cond1 = df_orders['order_purchase_timestamp'] > df_orders['order_approved_at']               # 주문 > 승인
cond2 = df_orders['order_approved_at'] > df_orders['order_delivered_carrier_date']           # 승인 > 출발
cond3 = df_orders['order_delivered_carrier_date'] > df_orders['order_delivered_customer_date'] # 출발 > 수령
cond4 = df_orders['order_purchase_timestamp'] > df_orders['order_delivered_customer_date']   # 주문 > 수령


In [71]:
# ── 이상치 확인 ──
print(f"주문 > 승인:   {cond1.sum()}건")
print(f"승인 > 출발:   {cond2.sum()}건")
print(f"출발 > 수령:   {cond3.sum()}건")
print(f"주문 > 수령:   {cond4.sum()}건")  

주문 > 승인:   0건
승인 > 출발:   1359건
출발 > 수령:   23건
주문 > 수령:   0건


In [72]:
# invalid_df 생성 (cond2 이상치)
invalid_df_2 = df_orders[cond2].copy()

In [73]:
print("\n=== order_status 분포 ===")
print(invalid_df_2['order_status'].value_counts())


=== order_status 분포 ===
order_status
delivered    1350
shipped         9
Name: count, dtype: int64


In [74]:
# ── cond2 재정의 (approved > carrier, delivered/shipped만) ──
target_status = df_orders['order_status'].isin(['delivered', 'shipped'])
time_diff = (
    df_orders['order_approved_at'] - df_orders['order_delivered_carrier_date']
).dt.total_seconds() / 3600

cond2 = target_status & (time_diff > 0)
df_orders['approval_vs_carrier_hours'] = time_diff  # 분석용 임시 컬럼

print(f"역전 건수 (cond2): {cond2.sum():,}건")
print("\n상태별 분포:")
print(df_orders[cond2]['order_status'].value_counts())

역전 건수 (cond2): 1,359건

상태별 분포:
order_status
delivered    1350
shipped         9
Name: count, dtype: int64


In [75]:
print("=== cond2 대상 행의 결측값 ===")
print(df_orders[cond2][['order_approved_at', 'order_delivered_carrier_date']].isnull().sum())
print("\n→ 모두 0이면, null이 아닌 진짜 역전 데이터")

=== cond2 대상 행의 결측값 ===
order_approved_at               0
order_delivered_carrier_date    0
dtype: int64

→ 모두 0이면, null이 아닌 진짜 역전 데이터


In [76]:
# cond2는 이미 정의되어 있으므로 status 조건만 추가
df_cond2_delivered = df_orders[cond2 & (df_orders['order_status'] == 'delivered')].copy()
df_cond2_shipped   = df_orders[cond2 & (df_orders['order_status'] == 'shipped')].copy()

print(f"cond2 delivered: {len(df_cond2_delivered):,}건")
print(f"cond2 shipped:   {len(df_cond2_shipped):,}건")

cond2 delivered: 1,350건
cond2 shipped:   9건


### cond2 (delivered)

In [77]:
pay_cond2_del = df_order_payments[df_order_payments['order_id'].isin(df_cond2_delivered['order_id'])]

print("=== 전체 결제 수단 비율 ===")
print((df_order_payments['payment_type'].value_counts(normalize=True) * 100).round(1))
print("\n=== cond2 delivered 결제 수단 비율 ===")
print((pay_cond2_del['payment_type'].value_counts(normalize=True) * 100).round(1))

=== 전체 결제 수단 비율 ===
payment_type
credit_card    73.9
boleto         19.0
voucher         5.6
debit_card      1.5
not_defined     0.0
Name: proportion, dtype: float64

=== cond2 delivered 결제 수단 비율 ===
payment_type
credit_card    72.0
boleto         20.4
voucher         4.4
debit_card      3.1
Name: proportion, dtype: float64


In [78]:
bins   = [0, 24, 72, 200, float('inf')]
labels = ['0~24h', '24~72h', '72~200h', '200h+']
df_cond2_delivered['time_bucket'] = pd.cut(
    df_cond2_delivered['approval_vs_carrier_hours'], bins=bins, labels=labels
)

print("=== 시간 구간별 건수 ===")
print(df_cond2_delivered['time_bucket'].value_counts().sort_index())

# 구간별 order_id 추출 후 결제 수단 비율
print("\n=== 시간 구간별 결제 수단 비율 (%) ===")
for bucket in labels:
    ids = df_cond2_delivered[df_cond2_delivered['time_bucket'] == bucket]['order_id']
    pay = df_order_payments[df_order_payments['order_id'].isin(ids)]
    if len(pay) == 0:
        continue
    ratio = (pay['payment_type'].value_counts(normalize=True) * 100).round(1)
    print(f"\n[{bucket}]")
    print(ratio.to_string())

=== 시간 구간별 건수 ===
time_bucket
0~24h      897
24~72h     360
72~200h     82
200h+       11
Name: count, dtype: int64

=== 시간 구간별 결제 수단 비율 (%) ===

[0~24h]
payment_type
credit_card    78.8
boleto         14.9
voucher         3.7
debit_card      2.6

[24~72h]
payment_type
credit_card    63.1
boleto         30.4
voucher         3.3
debit_card      3.3

[72~200h]
payment_type
credit_card    38.1
boleto         37.1
voucher        16.5
debit_card      8.2

[200h+]
payment_type
credit_card    100.0


In [79]:
# 요일 추출 (0=월, 6=일)
day_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'}

df_cond2_delivered['approved_dow']  = df_cond2_delivered['order_approved_at'].dt.dayofweek.map(day_map)
df_cond2_delivered['carrier_dow']   = df_cond2_delivered['order_delivered_carrier_date'].dt.dayofweek.map(day_map)

order_dow = ['월', '화', '수', '목', '금', '토', '일']

print("=== 시간 구간별 요일 분포 ===")
for bucket in labels:
    sub = df_cond2_delivered[df_cond2_delivered['time_bucket'] == bucket]
    if len(sub) == 0:
        continue
    print(f"\n[{bucket}] ({len(sub)}건)")
    print("  승인일 요일:")
    print(sub['approved_dow'].value_counts().reindex(order_dow).dropna().to_string())
    print("  배송 출고일 요일:")
    print(sub['carrier_dow'].value_counts().reindex(order_dow).dropna().to_string())

=== 시간 구간별 요일 분포 ===

[0~24h] (897건)
  승인일 요일:
approved_dow
월     61.0
화    383.0
수     93.0
목    285.0
금     68.0
토      7.0
  배송 출고일 요일:
carrier_dow
월    247.0
화    216.0
수     81.0
목    284.0
금     65.0
토      4.0

[24~72h] (360건)
  승인일 요일:
approved_dow
월      1
화     62
수     22
목    253
금     10
토      8
일      4
  배송 출고일 요일:
carrier_dow
월     68.0
화     90.0
수    188.0
목      9.0
금      5.0

[72~200h] (82건)
  승인일 요일:
approved_dow
월    12
화    11
수     5
목     6
금    14
토    18
일    16
  배송 출고일 요일:
carrier_dow
월    10.0
화    35.0
수    18.0
목     6.0
금     8.0
토     5.0

[200h+] (11건)
  승인일 요일:
approved_dow
월     1.0
수    10.0
  배송 출고일 요일:
carrier_dow
월    8.0
화    2.0
금    1.0


주말 비중이 많지 않다 -> 주말로 인한 승인 처리 지연이라고 할 수 없다
-> 승인 지연이 발생할 수 있는 24시간만 허용하도록 하여 승인과 배송 시간을 동일하게 보정하여 유지한다
-> 24시간을 초과한 행들은 이상치로 처리하여 삭제한다

In [80]:
mask_0_24h = cond2 & (df_orders['order_status'] == 'delivered') & (df_orders['approval_vs_carrier_hours'] <= 24)
df_orders.loc[mask_0_24h, 'order_approved_at'] = df_orders.loc[mask_0_24h, 'order_delivered_carrier_date']

print(f"보정 완료: {mask_0_24h.sum():,}건")

보정 완료: 897건


In [81]:
mask_over_24h = cond2 & (df_orders['order_status'] == 'delivered') & (df_orders['approval_vs_carrier_hours'] > 24)

print(f"삭제 대상: {mask_over_24h.sum():,}건")
df_orders = df_orders[~mask_over_24h]  
print(f"삭제 후 df_orders: {len(df_orders):,}건")

삭제 대상: 453건
삭제 후 df_orders: 97,751건


In [82]:
df_orders = df_orders.drop(columns=['approval_vs_carrier_hours'])

### cond2 (shipped)

In [83]:
print(df_cond2_shipped[[
    'order_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]])

                               order_id order_purchase_timestamp  \
4113   dee6298ce7d1fb2645141ef9972157aa      2018-04-30 14:06:12   
39196  ef18f5dcc7b8a44cbdb1a14de4164bd1      2018-07-04 10:52:38   
40483  388c070ce42d3ccaedfd15662e09631d      2018-04-21 03:25:21   
52958  e9ce8d3f379bf3d70e4af85174e1841f      2017-07-29 14:44:04   
67413  0749c820395dd586556efd230c9d1bb9      2018-07-04 13:12:48   
74152  986dfd5411cb5a65f3fe024bdb0d0745      2018-07-03 19:59:42   
79131  d29a401d12ff3433597a0aa1d17d37ba      2018-04-20 20:14:41   
79385  304e8cf62306a897106fd54ccfcc88bc      2018-04-20 11:58:38   
92746  cad95e56b8bc5ed2c10ee964ba21ac96      2018-01-30 14:44:20   

        order_approved_at order_delivered_carrier_date  \
4113  2018-04-30 14:15:24          2018-04-30 12:59:00   
39196 2018-07-05 16:32:54          2018-07-04 14:18:00   
40483 2018-04-24 18:58:14          2018-04-23 17:31:51   
52958 2017-08-02 08:34:47          2017-08-01 18:12:05   
67413 2018-07-05 16:25:09    

In [84]:
df_cond2_shipped = df_orders[cond2 & (df_orders['order_status'] == 'shipped')].copy()
print(f"cond2 shipped 건수: {len(df_cond2_shipped):,}건")

shipped_review = df_order_reviews_clean[df_order_reviews_clean['order_id'].isin(df_cond2_shipped['order_id'])]

print("\n=== 평점별 건수 & 리뷰 샘플 ===")
for score in sorted(shipped_review['review_score'].dropna().unique()):
    subset = shipped_review[shipped_review['review_score'] == score]
    print(f"\n[{int(score)}점] {len(subset)}건")


cond2 shipped 건수: 9건

=== 평점별 건수 & 리뷰 샘플 ===

[1점] 3건

[2점] 2건

[4점] 1건

[5점] 1건


/var/folders/6y/qvnf_sw97x1_0r6dbbkwmpc80000gn/T/ipykernel_69825/1264421255.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_cond2_shipped = df_orders[cond2 & (df_orders['order_status'] == 'shipped')].copy()


배송중인데 리뷰가 4점, 5점인 것은 이상하다. 4점, 5점인 것은 삭제 후 1점, 2점인 행들은 승인, 배송 시간을 동일하게 보정하여 유지한다

In [85]:
# ── 1~2점: 승인 시각을 배송 출고 시각으로 보정 ──
ids_score_1_2 = shipped_review[shipped_review['review_score'].isin([1, 2])]['order_id']
mask_fix = df_orders['order_id'].isin(ids_score_1_2)

df_orders.loc[mask_fix, 'order_approved_at'] = df_orders.loc[mask_fix, 'order_delivered_carrier_date']
print(f"보정 (1~2점): {mask_fix.sum()}건")

# ── 4~5점: 삭제 ──
ids_score_4_5 = shipped_review[shipped_review['review_score'].isin([4, 5])]['order_id']

print(f"삭제 (4~5점): {len(ids_score_4_5)}건")
df_orders = df_orders[~df_orders['order_id'].isin(ids_score_4_5)]
print(f"삭제 후 df_orders: {len(df_orders):,}건")

보정 (1~2점): 5건
삭제 (4~5점): 2건
삭제 후 df_orders: 97,749건


In [86]:
# 리뷰가 없는 shipped+cond2 order_id 추출
ids_with_review = shipped_review['order_id']
ids_no_review = df_cond2_shipped[~df_cond2_shipped['order_id'].isin(ids_with_review)]['order_id']

print(f"리뷰 없는 건수: {len(ids_no_review)}건")

# 보정: approved_at = carrier_date
mask_no_review = df_orders['order_id'].isin(ids_no_review)
df_orders.loc[mask_no_review, 'order_approved_at'] = df_orders.loc[mask_no_review, 'order_delivered_carrier_date']

print(f"보정 완료: {mask_no_review.sum()}건")

리뷰 없는 건수: 2건
보정 완료: 2건


In [87]:
# 중복 확인
print(f"shipped_review 전체 행수: {len(shipped_review)}건")
print(f"shipped_review 유니크 order_id: {shipped_review['order_id'].nunique()}건")


shipped_review 전체 행수: 7건
shipped_review 유니크 order_id: 7건


## cond3 

In [89]:
# cond3 재계산 (행 삭제 후 최신 df 기준)
cond3 = df_orders['order_delivered_carrier_date'] > df_orders['order_delivered_customer_date']

In [90]:
# invalid_df 생성 (cond3 이상치)
invalid_df_3 = df_orders[cond3].copy()

print("\n=== order_status 분포 ===")
print(invalid_df_3['order_status'].value_counts())


=== order_status 분포 ===
order_status
delivered    23
Name: count, dtype: int64
